In [ ]:
import sys
print([p for p in sys.path if 'spark' in p])

In [ ]:
import os
from pyspark.sql import SparkSession

SPARK_MASTER = os.environ.get('SPARK_MASTER', 'spark://spark-master:7077')

spark = SparkSession.builder \
    .appName('Demo Spark - Arquitecto de Datos') \
    .master(SPARK_MASTER) \
    .config('spark.executor.memory', '512m') \
    .config('spark.driver.host', 'jupyter') \
    .getOrCreate()

print(f'Spark version : {spark.version}')
print(f'Master        : {spark.sparkContext.master}')
print(f'App ID        : {spark.sparkContext.applicationId}')

In [ ]:
texto = [
    'hadoop spark kafka flink',
    'spark es rapido spark es potente',
    'kafka es distribuido kafka es escalable',
    'hadoop hdfs yarn mapreduce',
    'spark dataframe spark rdd spark streaming'
]

word_count = (
    spark.sparkContext.parallelize(texto)
    .flatMap(lambda line: line.split())
    .map(lambda word: (word, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
)

print('Top palabras:')
for word, count in word_count.take(10):
    print(f'  {word:20s} {count}')

In [ ]:
from pyspark.sql import Row
from pyspark.sql.functions import avg, count, round as spark_round, col

datos = [
    Row(region='Norte', producto='A', importe=1200.0, unidades=10),
    Row(region='Norte', producto='B', importe=800.0,  unidades=5),
    Row(region='Sur',   producto='A', importe=950.0,  unidades=8),
    Row(region='Sur',   producto='C', importe=1500.0, unidades=12),
    Row(region='Este',  producto='B', importe=600.0,  unidades=4),
    Row(region='Este',  producto='A', importe=1100.0, unidades=9),
    Row(region='Oeste', producto='C', importe=2000.0, unidades=15),
    Row(region='Oeste', producto='A', importe=750.0,  unidades=6),
]

df = spark.createDataFrame(datos)
df.printSchema()
df.show()

In [ ]:
resumen = df.groupBy('region').agg(
    spark_round(avg('importe'), 2).alias('importe_medio'),
    count('*').alias('num_ventas'),
    spark_round(avg('unidades'), 1).alias('unidades_media')
).orderBy(col('importe_medio').desc())

resumen.show()

In [ ]:
df.createOrReplaceTempView('ventas')

spark.sql("""
    SELECT producto,
           SUM(importe)  AS total_importe,
           SUM(unidades) AS total_unidades,
           COUNT(*)      AS num_regiones
    FROM ventas
    GROUP BY producto
    ORDER BY total_importe DESC
""").show()

In [ ]:
spark.sql("SELECT region, SUM(importe) FROM ventas GROUP BY region").explain(mode='formatted')

In [ ]:
spark.stop()
print('Sesión Spark cerrada.')